In [1]:
# ① 场景 YAML → ScenarioConfig（仅解析，尚未执行 Agent）
from arl.sandbox.config import ScenarioConfig

SCENARIO = "configs/scenarios/p1-sandbox-5step-direct-api.yaml"
cfg = ScenarioConfig.from_yaml(SCENARIO)

print("experiment_id:", cfg.experiment_id)
print("framework:    ", cfg.agent.framework)
print("model:        ", cfg.agent.model)
print("max_steps:    ", cfg.runner.max_steps)
print("tools.mode:   ", cfg.tools.mode)
print("seed:         ", cfg.reproducibility.seed)

5
42


In [ ]:
# ② LLM 录像带路径（回放模式无需 OPENAI_API_KEY）
from arl.sandbox.config import cassette_path

cp = cassette_path(cfg)
print("cassette:", cp)
print("存在:    ", cp.exists())

In [ ]:
# ③ 完整执行：session + DirectAPIBackend + mock 工具 + cassette 回放 → trajectory
from arl.sandbox.run import run_scenario

traj = run_scenario(SCENARIO)

print("session_id:", traj["session_id"])
print("framework: ", traj["framework"])
print("步数:      ", len(traj["steps"]))
print()

for i, s in enumerate(traj["steps"]):
    st = s["step_type"]
    if st == "llm_inference":
        llm = s["llm"]
        intents = llm.get("tool_call_intents") or []
        if intents:
            names = [x["name"] for x in intents]
            print(f"[{i}] LLM turn={llm['turn_index']} -> 要调工具: {names}")
        else:
            print(f"[{i}] LLM turn={llm['turn_index']} -> 纯文本结束")
    else:
        tc = s["tool_call"]
        print(f"[{i}] TOOL {tc['name']} -> {tc['response']}")